In [1]:
import arviz as az
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from scipy.stats import norm
import pandas as pd
import geopandas as gpd
import pymc as pm
import pymc_bart as pmb
import nutpie
import pytensor.tensor as pt
import os

SAFE_EXP_MAX = np.log(np.finfo(np.float64).max)
BIG = np.finfo(np.float64).max / 10

In [2]:
def safe_exp(x, big=BIG):
    """
    Compute exp(x) safely:
    - Clip x to <= SAFE_EXP_MAX to avoid overflow.
    - Replace non-finite outputs (NaN/Inf) with a large sentinel 'big'.
    """
    x = np.asarray(x, dtype=np.float64)
    # Clip only upper bound to prevent overflow; underflow to ~0 is fine for many tasks
    x_clipped = np.clip(x, None, SAFE_EXP_MAX)
    with np.errstate(over='ignore', invalid='ignore', under='ignore'):
        y = np.exp(x_clipped)
    # Replace any remaining non-finite results with BIG
    y = np.where(np.isfinite(y), y, big)
    return y

def make_doubly_robust_adjustment(y_pred, y, i, ips):
    
    dr_estimate = y_pred + ips * (y - y_pred)

    return dr_estimate

def get_ate(trace, t, y, ips, i):
    ### Post processing the sample posterior distribution for propensity scores
    ### One sample at a time.
    y_pred = trace["posterior"]["y_hat"].stack(z=("chain", "draw"))[:, i].values

    dr_estimate = make_doubly_robust_adjustment(
        y_pred, y, i, ips
    )
    
    bandwidth = 0.025 * (np.max(t) - np.min(t))  # Example: 5% of range
    
    t_high, t_low = np.percentile(t, [75, 25])
    
    trt = np.mean(dr_estimate[(t >= t_high - bandwidth) & (t <= t_high + bandwidth)])
    ntrt = np.mean(dr_estimate[(t >= t_low - bandwidth) & (t <= t_low + bandwidth)])

    etrt = safe_exp(trt)
    entrt = safe_exp(ntrt)
    
    ate = etrt - entrt
    if i%500==0:
        print("Done", i)
        # print("trt",trt)
        # print("ntrt",ntrt)
    return [ate, trt, ntrt]

def plot_ate(ate_dist_df, model, xy=(4.0, 250)):
    # 0.45 * textwidth on A4 (~6.14 in)
    fig_width = 0.4 * 6.14  # ≈ 2.76 inches
    fig_height = fig_width * 0.75  # aesthetic aspect ratio

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    # Calculate and plot mean line
    ate = np.round(ate_dist_df["ATE"].mean(), 2)
    ax.hist(ate_dist_df["ATE"], bins=30, ec="black", color="slateblue", label="ATE", alpha=0.6)
    ax.axvline(ate, label="E(ATE)", linestyle="--", color="black")
    
    # Annotate the mean
   # ax.annotate(f"E(ATE): {ate}", xy, fontsize=20, fontweight="bold")
    
    # Titles and labels
    ax.set_title(
    f"E(ATE): {ate}",
    fontsize=12)
    ax.set_xlabel("tFFCO2 per capita")
    ax.set_ylabel("Posterior frequency")
    ax.legend()

    fig.savefig(f"/work/hawkins_lab/vulcan/results/vulcan-{model}-pie.pdf", format="pdf", bbox_inches="tight")
    plt.close(fig)  # Closes the figure to free memory

# Transportation

In [3]:
tran_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_ONR_epa_climate.parquet'
tran_df = gpd.read_parquet(tran_par_file)
tran_wgt_file = '/work/hawkins_lab/vulcan/results/tran_weights.csv'
tran_wgt_df = pd.read_csv(tran_wgt_file)
tran_df = pd.concat([tran_df,tran_wgt_df], axis=1)
tran_df = tran_df[(tran_df["value_weig"]>0) & (tran_df["totpop"]>0)]
tran_df['d4a'] = tran_df['d4a'].replace(-99999,1500)

y = tran_df["value_weig"] / tran_df["totpop"].replace(0, np.nan) # make per capita
y = np.log(y)
y.isna().sum()

np.int64(0)

In [4]:
trace = az.from_netcdf("/work/hawkins_lab/Mehrnoosh/vulcan-results/vulcan-results 12-08/vulcan-tran_dens-pie.nc")
ips = 1/tran_df["dens"]

qs = range(4000)
ate_dist = [get_ate(trace, tran_df['d1a'], y, ips, q) for q in qs]

ate_dist_df = pd.DataFrame(ate_dist, columns=["ATE", "E(Y(75per))", "E(Y(25per))"])
ate_dist_df.head()

plot_ate(ate_dist_df, "tran_dens")

Done 0
Done 500
Done 1000
Done 1500
Done 2000
Done 2500
Done 3000
Done 3500


In [5]:
trace = az.from_netcdf("/work/hawkins_lab/Mehrnoosh/vulcan-results/vulcan-results 12-08/vulcan-tran_div-pie.nc")
ips = 1/tran_df["div"]

qs = range(4000)
ate_dist = [get_ate(trace, tran_df['d2b_e5mixa'], y, ips, q) for q in qs]

ate_dist_df = pd.DataFrame(ate_dist, columns=["ATE", "E(Y(75per))", "E(Y(25per))"])
ate_dist_df.head()

plot_ate(ate_dist_df, "tran_div")

Done 0
Done 500
Done 1000
Done 1500
Done 2000
Done 2500
Done 3000
Done 3500


In [6]:
trace = az.from_netcdf("/work/hawkins_lab/Mehrnoosh/vulcan-results/vulcan-results 12-08/vulcan-tran_des-pie.nc")
ips = 1/tran_df["des"]

qs = range(4000)
ate_dist = [get_ate(trace, tran_df['d3a'], y, ips, q) for q in qs]

ate_dist_df = pd.DataFrame(ate_dist, columns=["ATE", "E(Y(75per))", "E(Y(25per))"])
ate_dist_df.head()

plot_ate(ate_dist_df, "tran_des")

Done 0
Done 500
Done 1000
Done 1500
Done 2000
Done 2500
Done 3000
Done 3500


In [7]:
trace = az.from_netcdf("/work/hawkins_lab/Mehrnoosh/vulcan-results/vulcan-results 02-22-2026/vulcan-tran_dist-pie.nc")
ips = 1/tran_df["dist"]

qs = range(4000)
ate_dist = [get_ate(trace, tran_df['d4a'], y, ips, q) for q in qs]

ate_dist_df = pd.DataFrame(ate_dist, columns=["ATE", "E(Y(75per))", "E(Y(25per))"])
ate_dist_df.head()

plot_ate(ate_dist_df, "tran_dist")

Done 0
Done 500
Done 1000
Done 1500
Done 2000
Done 2500
Done 3000
Done 3500


In [8]:
trace = az.from_netcdf("/work/hawkins_lab/Mehrnoosh/vulcan-results/vulcan-results 12-08/vulcan-tran_dest-pie.nc")
ips = 1/tran_df["dest"]

qs = range(4000)
ate_dist = [get_ate(trace, tran_df['d5ar'], y, ips, q) for q in qs]

ate_dist_df = pd.DataFrame(ate_dist, columns=["ATE", "E(Y(75per))", "E(Y(25per))"])
ate_dist_df.head()

plot_ate(ate_dist_df, "tran_dest")

Done 0
Done 500
Done 1000
Done 1500
Done 2000
Done 2500
Done 3000
Done 3500


# Residential Electricity

In [9]:
elec_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_SC2_epa_climate.parquet'
elec_df = gpd.read_parquet(elec_par_file)
elec_wgt_file = '/work/hawkins_lab/vulcan/results/elec_weights.csv'
elec_wgt_df = pd.read_csv(elec_wgt_file)
elec_df = pd.concat([elec_df,elec_wgt_df], axis=1)
elec_df = elec_df[(elec_df["res_tc"]>0) & (elec_df["totpop"]>0)]

y = elec_df["res_tc"] / elec_df["totpop"].replace(0, np.nan) # make per capita
y = np.log(y)
y.isna().sum()

np.int64(0)

In [10]:
trace = az.from_netcdf("/work/hawkins_lab/Mehrnoosh/vulcan-results/vulcan-results 02-22-2026/vulcan-elec_dens-pie.nc")
ips = 1/elec_df["dens"]

qs = range(4000)
ate_dist = [get_ate(trace, elec_df['d1a'], y, ips, q) for q in qs]

ate_dist_df = pd.DataFrame(ate_dist, columns=["ATE", "E(Y(75per))", "E(Y(25per))"])
ate_dist_df.head()

plot_ate(ate_dist_df, "elec_dens")

Done 0
Done 500
Done 1000
Done 1500
Done 2000
Done 2500
Done 3000
Done 3500


In [11]:
trace = az.from_netcdf("/work/hawkins_lab/Mehrnoosh/vulcan-results/vulcan-results 02-22-2026/vulcan-elec_div-pie.nc")
ips = 1/elec_df["div"]

qs = range(4000)
ate_dist = [get_ate(trace, elec_df['d2b_e5mixa'], y, ips, q) for q in qs]

ate_dist_df = pd.DataFrame(ate_dist, columns=["ATE", "E(Y(75per))", "E(Y(25per))"])
ate_dist_df.head()

plot_ate(ate_dist_df, "elec_div")

Done 0
Done 500
Done 1000
Done 1500
Done 2000
Done 2500
Done 3000
Done 3500


# Residential Energy

In [12]:
res_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_RES_epa_climate.parquet'
res_df = gpd.read_parquet(res_par_file)
res_wgt_file = '/work/hawkins_lab/vulcan/results/res_weights.csv'
res_wgt_df = pd.read_csv(res_wgt_file)
res_df = pd.concat([res_df,res_wgt_df], axis=1)
res_df = res_df[(res_df["value_weig"]>0) & (res_df["totpop"]>0)]

y = res_df["value_weig"] / res_df["totpop"].replace(0, np.nan) # make per capita
y = np.log(y)
y.isna().sum()

np.int64(0)

In [13]:
trace = az.from_netcdf("/work/hawkins_lab/Mehrnoosh/vulcan-results/vulcan-results 12-08/vulcan-res_dens-pie.nc")
ips = 1/res_df["dens"]

qs = range(4000)
ate_dist = [get_ate(trace, res_df['d1a'], y, ips, q) for q in qs]

ate_dist_df = pd.DataFrame(ate_dist, columns=["ATE", "E(Y(75per))", "E(Y(25per))"])
ate_dist_df.head()

plot_ate(ate_dist_df, "res_dens")

Done 0
Done 500
Done 1000
Done 1500
Done 2000
Done 2500
Done 3000
Done 3500


In [ ]:
trace = az.from_netcdf("/work/hawkins_lab/Mehrnoosh/vulcan-results/vulcan-results 12-08/vulcan-res_div-pie.nc")
ips = 1/res_df["div"]

qs = range(4000)
ate_dist = [get_ate(trace, res_df['d2b_e5mixa'], y, ips, q) for q in qs]

ate_dist_df = pd.DataFrame(ate_dist, columns=["ATE", "E(Y(75per))", "E(Y(25per))"])
ate_dist_df.head()

plot_ate(ate_dist_df, "res_div")